<a href="https://colab.research.google.com/github/udikakumari2/Programming-assignment-02/blob/main/PA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dataset Description**

---

This research uses an artificial dataset compiled by me for detecting the players and keypoints from sports videos. This study involves 9 sports video clips collected from public online resources and arranged in the dataset. The video clips contain scenes from sports activities with different challenging situations such as many players, various camera perspectives, player actions, and sports environments.

This dataset includes several categories of sports videos such as football, rugby, basketball, and plays performed by students. All the video clips have 5-10 second lengths which meet the criteria of the assignment. The clips were saved in MP4 format and also included metadata with video ID, name, sport category, and download link.

The main purpose of creation this dataset is to test the performance of the YOLOv8-based player detection model and YOLOv8 pose estimation (keypoints detection). This dataset can be used as a benchmark for evaluating the performance of object detection and human pose estimation models.
| Attribute      | Description                    |
| -------------- | ------------------------------ |
| ID             | Unique identifier of the video |
| Name           | Video name                     |
| Sport Name     | Category of sport              |
| Link           | [link text](https://docs.google.com/spreadsheets/d/1Yx5mennlMv9ot2VZXMSDdVSM4-sIGI2g/edit?usp=sharing&ouid=113169337633899006557&rtpof=true&sd=true)       |
| Total Videos   | 9                              |
| Video Duration | 5–10 seconds                   |
| Video Format   | MP4                            |





# **1.Player Detection**

---





Build a vision model using a YOLO-like object detection framework to detect players in
the videos.

In [ ]:

!pip install ultralytics gdown -q

from ultralytics import YOLO
import cv2
import pandas as pd
import gdown
import os

model = YOLO("yolov8m.pt")


os.makedirs("output_videos", exist_ok=True)


df = pd.read_excel("Dataset For PA 2.xlsx")

def convert_drive_link(link):
    file_id = link.split('/d/')[1].split('/')[0]
    return f"https://drive.google.com/uc?id={file_id}"


for index, row in df.iterrows():

    video_name = str(row['Name'])
    video_link = row['Link']

    print(f"\nProcessing: {video_name}")


    download_link = convert_drive_link(video_link)
    input_video = f"{video_name}.mp4"

    gdown.download(download_link,
                   input_video,
                   quiet=False)


    cap = cv2.VideoCapture(input_video)

    if not cap.isOpened():
        print("Cannot open video")
        continue

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 30

    output_video = f"output_videos/{video_name}_players.mp4"

    out = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (width, height)
    )

    frame_count = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_count += 1

        results = model(frame, classes=[0])

        annotated = results[0].plot()


        out.write(annotated)

        if frame_count % 50 == 0:
            print(f"Processed {frame_count} frames")

    cap.release()
    out.release()

    print(f"Saved: {output_video}")

print("\nAll videos processed successfully!")

# **2.Keypoint Detection**





---


Implement a model similar to OpenPose to identify player keypoints (e.g., hands, legs,
body joints).

In [ ]:
# Install libraries
!pip install ultralytics gdown -q

from ultralytics import YOLO
import cv2
import pandas as pd
import gdown
import os

# Load YOLO Pose model
model = YOLO("yolov8m-pose.pt")

# Create output folder
os.makedirs("output_pose_videos", exist_ok=True)

# Read Excel file
df = pd.read_excel("Dataset For PA 2.xlsx")

def convert_drive_link(link):
    file_id = link.split('/d/')[1].split('/')[0]
    return f"https://drive.google.com/uc?id={file_id}"

# Process every video
for index, row in df.iterrows():

    video_name = str(row['Name'])
    video_link = row['Link']

    print(f"\nProcessing: {video_name}")

    # Download video
    download_link = convert_drive_link(video_link)
    input_video = f"{video_name}.mp4"

    gdown.download(download_link,
                   input_video,
                   quiet=False)

    # Open video
    cap = cv2.VideoCapture(input_video)

    if not cap.isOpened():
        print("Cannot open video")
        continue

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 30

    output_video = f"output_pose_videos/{video_name}_pose.mp4"

    out = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (width, height)
    )

    frame_count = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_count += 1

        # Run pose estimation
        results = model(frame)

        # Draw keypoints and skeleton automatically
        annotated_frame = results[0].plot()

        # Save frame
        out.write(annotated_frame)

        if frame_count % 50 == 0:
            print(f"Processed {frame_count} frames")

    cap.release()
    out.release()

    print(f"Saved: {output_video}")

print("\nAll videos processed successfully!")